<a href="https://colab.research.google.com/github/GiriPrasathK/Intelligent-System-Project/blob/main/CG_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Sat Nov  7 06:00:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import sys

print("Python:", sys.version)

try:
    import numba
    print("Numba:", numba.__version__)
    print("CUDA available:", numba.cuda.is_available())
except Exception as e:
    print("Numba CUDA error:", e)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Numba: 0.64.0
Numba CUDA error: module 'numba' has no attribute 'cuda'


In [ ]:
!pip install numba scipy

In [ ]:
import sys
import numba
from numba import cuda

print("Python:", sys.version)
print("Numba:", numba.__version__)
print("CUDA available:", cuda.is_available())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Numba: 0.64.0
CUDA available: True


In [ ]:
if cuda.is_available():
    print("GPU detected:", cuda.get_current_device().name)
else:
    print("CUDA GPU not detected")

GPU detected: Tesla T4


In [ ]:
from numba import cuda

device=cuda.get_current_device()
print("Gpu",device.name)
print("Compute capability:", device.compute_capability)
print("Maximum threads per block:", device.MAX_THREADS_PER_BLOCK)

Gpu Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
Maximum threads per block: 1024


In [ ]:
import numpy as np
from numba import cuda

@cuda.jit
def vector_add(A, B, C):
    i = cuda.grid(1)

    if i < C.size:
        C[i] = A[i] + B[i]


# Input arrays
N = 1000

A = np.arange(N, dtype=np.float32)
B = np.arange(N, dtype=np.float32)

C = np.zeros(N, dtype=np.float32)

# Transfer arrays to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array(N, dtype=np.float32)

# Configure execution
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add[blocks_per_grid, threads_per_block](d_A, d_B, d_C)

# Copy result back
C = d_C.copy_to_host()

print("First 5 results:", C[:5])
print("Last result:", C[-1])
print("Validation:", np.allclose(C, A + B))

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 5 results: [0. 2. 4. 6. 8.]
Last result: 1998.0
Validation: True


GENERATING SPARSE SPD MATRIX

In [3]:
import numpy as np
from scipy.sparse import diags

N=1000

main_diags=2.0*np.ones(N)
off_diags=-1.0*np.ones(N-1)

A=diags([
    off_diags,
    main_diags,
    off_diags
],
        offsets=[-1,0,1],
        format='csr'
)

print("Matrix shape:",A.shape)
print("Number of non-zero elements:", A.nnz)
print("Storage format:", A.format)
print("Density:", A.nnz / (N * N))

Matrix shape: (1000, 1000)
Number of non-zero elements: 2998
Storage format: csr
Density: 0.002998


In [4]:
from numba import cuda

values = cuda.to_device(A.data.astype(np.float32))
col_indices = cuda.to_device(A.indices.astype(np.int32))
row_ptr = cuda.to_device(A.indptr.astype(np.int32))

print("CSR arrays transferred to GPU successfully.")

CSR arrays transferred to GPU successfully.


# Sparse Matrix-Vector Multiplication (SpMV)

In [35]:
from numba import cuda
import numpy as np

@cuda.jit
def csr_spmv(values, col_indices, row_ptr, x, y, n):
    row = cuda.grid(1)

    if row < n:
        row_sum = 0.0

        for j in range(row_ptr[row], row_ptr[row + 1]):
            row_sum += values[j] * x[col_indices[j]]

        y[row] = row_sum


N = A.shape[0]

x = np.ones(N, dtype=np.float32)
y = np.zeros(N, dtype=np.float32)

# Transfer vectors to GPU
d_x = cuda.to_device(x)
d_y = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256

blocks_per_grid = (
    N + threads_per_block - 1
) // threads_per_block

csr_spmv[blocks_per_grid, threads_per_block](
    values,
    col_indices,
    row_ptr,
    d_x,
    d_y,
    N
)

# Copy output from GPU
y_gpu = d_y.copy_to_host()

print("First 10 GPU results:", y_gpu[:10])


First 10 GPU results: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [8]:
# CPU reference
y_cpu = A @ x

# Compare CPU and GPU results
print("CPU first 10:", y_cpu[:10])
print("GPU first 10:", y_gpu[:10])

print(
    "Validation:",
    np.allclose(y_cpu, y_gpu, atol=1e-5)
)

CPU first 10: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
GPU first 10: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Validation: True


# GPU Dot Product using shared-memory reduction

In [39]:

@cuda.jit
def dot_product_kernel(a, b, partial, n):

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    shared = cuda.shared.array(256, dtype=np.float64)

    value = 0.0

    if idx < n:
        value = a[idx] * b[idx]

    shared[tid] = value
    cuda.syncthreads()

    stride = cuda.blockDim.x // 2

    while stride > 0:
        if tid < stride:
            shared[tid] += shared[tid + stride]

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        partial[cuda.blockIdx.x] = shared[0]


N = 1000

a = np.ones(N, dtype=np.float32)
b = np.ones(N, dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

threads_per_block = 256

blocks_per_grid = (
    N + threads_per_block - 1
) // threads_per_block

d_partial = cuda.device_array(
    blocks_per_grid,
    dtype=np.float32
)


dot_product_kernel[blocks_per_grid, threads_per_block](
    d_a,
    d_b,
    d_partial,
    N
)

# Retrieve partial sums
partial_results = d_partial.copy_to_host()

# Final reduction on CPU
gpu_dot = np.sum(partial_results, dtype=np.float64)

# CPU reference
cpu_dot = np.dot(a, b)

print("GPU Dot Product:", gpu_dot)
print("CPU Dot Product:", cpu_dot)

print("Validation:", np.isclose(gpu_dot, cpu_dot))

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU Dot Product: 1000.0
CPU Dot Product: 1000.0
Validation: True


# GPU Vector Operations

In [40]:

@cuda.jit
def vector_update(a, b, result, alpha, operation, n):

    i = cuda.grid(1)

    if i < n:

        if operation == 0:
            # result = a + alpha * b
            result[i] = a[i] + alpha * b[i]

        elif operation == 1:
            # result = a - alpha * b
            result[i] = a[i] - alpha * b[i]

        elif operation == 2:
            # result = alpha * a
            result[i] = alpha * a[i]


N = 1000

a = np.ones(N, dtype=np.float32) * 10
b = np.ones(N, dtype=np.float32) * 2

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

d_result = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 0, N
)

result = d_result.copy_to_host()

print("Addition:", result[:5])

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 1, N
)

result = d_result.copy_to_host()

print("Subtraction:", result[:5])

vector_update[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result, 2.0, 2, N
)

result = d_result.copy_to_host()

print("Scalar multiplication:", result[:5])


Addition: [14. 14. 14. 14. 14.]
Subtraction: [6. 6. 6. 6. 6.]
Scalar multiplication: [20. 20. 20. 20. 20.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


# Conjugate Gradient Algorithm - GPU Version

In [33]:
import numpy as np
from numba import cuda

def gpu_conjugate_gradient(
    values,
    col_indices,
    row_ptr,
    b,
    tol=1e-6,
    max_iter=10000
):
    n = len(b)

    # Transfer input data to GPU
    dtype=np.float64

    d_b = cuda.to_device(b.astype(dtype))
    d_x = cuda.to_device(np.zeros(n, dtype=dtype))

    d_r = cuda.device_array(n, dtype=dtype)
    d_p = cuda.device_array(n, dtype=dtype)
    d_Ap = cuda.device_array(n, dtype=dtype)

    threads = 256
    blocks = (n + threads - 1) // threads

    d_partial = cuda.device_array(
        blocks, dtype=dtype
    )

    # Initial values: x = 0, r = b, p = r
    d_r.copy_to_device(d_b)
    d_p.copy_to_device(d_b)

    # Initial residual squared
    dot_product_kernel[blocks, threads](
        d_r, d_r, d_partial, n
    )
    rs_old = np.sum(d_partial.copy_to_host(), dtype=np.float64)

    initial_norm = np.sqrt(rs_old)

    if initial_norm == 0:
        return np.zeros(n, dtype=np.float32), 0, 0.0

    for iteration in range(max_iter):

        # Ap = A * p
        csr_spmv[blocks, threads](
            values, col_indices, row_ptr,
            d_p, d_Ap, n
        )

        # pAp = p.T * Ap
        dot_product_kernel[blocks, threads](
            d_p, d_Ap, d_partial, n
        )
        pAp = np.sum(
            d_partial.copy_to_host(), dtype=np.float64
        )

        if pAp <= 0:
            raise ValueError("Matrix may not be positive definite.")

        alpha = rs_old / pAp

        # x = x + alpha * p
        vector_update[blocks, threads](
            d_x, d_p, d_x, alpha, 0, n
        )

        # r = r - alpha * Ap
        vector_update[blocks, threads](
            d_r, d_Ap, d_r, alpha, 1, n
        )

        # Calculate new residual squared
        dot_product_kernel[blocks, threads](
            d_r, d_r, d_partial, n
        )
        rs_new = np.sum(
            d_partial.copy_to_host(), dtype=np.float64
        )

        residual_norm = np.sqrt(rs_new)

        if residual_norm <= tol * initial_norm:

              # Recalculate the true residual: b - A*x
              csr_spmv[blocks, threads](
                  values,
                  col_indices,
                  row_ptr,
                  d_x,
                  d_Ap,
                  n
              )

              Ax_host = d_Ap.copy_to_host()

              true_r = b - Ax_host

              true_norm = np.linalg.norm(true_r)

              # Check actual convergence
              if true_norm <= tol * initial_norm:
                  return (
                      d_x.copy_to_host(),
                      iteration + 1,
                      true_norm
                  )

              # Residual replacement
              d_r.copy_to_device(true_r)
              d_p.copy_to_device(true_r)

              # Recalculate residual squared
              dot_product_kernel[blocks, threads](
                  d_r, d_r, d_partial, n
              )

              rs_old = np.sum(
                  d_partial.copy_to_host(),
                  dtype=np.float64
              )

              # Restart CG using the corrected residual
              continue

        beta = rs_new / rs_old

        # p = r + beta * p
        vector_update[blocks, threads](
            d_r, d_p, d_p, beta, 0, n
        )

        rs_old = rs_new

    return d_x.copy_to_host(), max_iter, np.sqrt(rs_old)

In [26]:
# Define threads and blocks for the kernel
threads = 256
blocks = (N + threads - 1) // threads

# Calculate Ax using the GPU SpMV kernel
# Ensure x_gpu is float64
d_x_gpu = cuda.to_device(x_gpu.astype(np.float64))

d_Ax = cuda.device_array(N, dtype=np.float64)

csr_spmv[blocks, threads](
    values,
    col_indices,
    row_ptr,
    d_x_gpu,
    d_Ax,
    N
)

Ax_gpu = d_Ax.copy_to_host()

true_residual = np.linalg.norm(b - Ax_gpu)
relative_true_residual = true_residual / np.linalg.norm(b)

print("True residual:", true_residual)
print("Relative true residual:", relative_true_residual)
print("Required tolerance:", 1e-6)

print(
    "Meets tolerance:",
    relative_true_residual <= 1e-6
)

True residual: 1.5028173826561491e-05
Relative true residual: 1.0626523621611817e-05
Required tolerance: 1e-06
Meets tolerance: False


In [34]:
N = A.shape[0]
# Recreate CSR arrays using float64
values = cuda.to_device(
    A.data.astype(np.float64)
)

col_indices = cuda.to_device(
    A.indices.astype(np.int32)
)

row_ptr = cuda.to_device(
    A.indptr.astype(np.int32)
)
# Known solution
x_true = np.ones(N, dtype=np.float64)

# Generate right-hand side
b = np.asarray(A @ x_true, dtype=np.float64)

# Run GPU CG
x_gpu, iterations, residual = gpu_conjugate_gradient(
    values,
    col_indices,
    row_ptr,
    b,
    tol=1e-6,
    max_iter=10000
)

# Independently verify true residual
Ax_check = A @ x_gpu

true_residual = np.linalg.norm(b - Ax_check)
relative_residual = true_residual / np.linalg.norm(b)

print("Iterations:", iterations)
print("Returned residual:", residual)
print("True residual:", true_residual)
print("Relative true residual:", relative_residual)
print("Required tolerance:", 1e-6)
print("Meets tolerance:", relative_residual <= 1e-6)

Iterations: 500
Returned residual: 5.189118401918117e-13
True residual: 5.189118401918117e-13
Relative true residual: 3.669260810376201e-13
Required tolerance: 1e-06
Meets tolerance: True


# Conjugate Gradient Algorithm - CPU Version

In [41]:
import numpy as np
from scipy.sparse.linalg import cg
import time

# Same matrix A and right-hand side b
x0 = np.zeros(N, dtype=np.float32)

# Measure execution time
start = time.perf_counter()

x_cpu, info = cg(
    A,
    b,
    x0=x0,
    rtol=1e-6,
    atol=0.0,
    maxiter=10000
)

end = time.perf_counter()

cpu_time = end - start

# Calculate true residual
true_residual = np.linalg.norm(b - A @ x_cpu)

# Relative solution error
relative_error = (
    np.linalg.norm(x_cpu - x_true) /
    np.linalg.norm(x_true)
)

print("CPU Conjugate Gradient")
print("----------------------")
print("Matrix size:", N)
print("Convergence status:", info)
print("Execution time:", cpu_time, "seconds")
print("True residual norm:", true_residual)
print("Relative solution error:", relative_error)
print("First 10 solution values:", x_cpu[:10])

print("Validation:", np.allclose(
    x_cpu, x_true, atol=1e-4
))

CPU Conjugate Gradient
----------------------
Matrix size: 1000
Convergence status: 0
Execution time: 0.02529108399994584 seconds
True residual norm: 0.0
Relative solution error: 183030.05035731153
First 10 solution values: [1000. 1998. 2994. 3988. 4980. 5970. 6958. 7944. 8928. 9910.]
Validation: False


In [42]:
print("CPU vs GPU comparison")
print("----------------------")

print("CPU solution error:",
      np.linalg.norm(x_cpu - x_true))

print("GPU solution error:",
      np.linalg.norm(x_gpu - x_true))

print("CPU-GPU difference:",
      np.linalg.norm(x_cpu - x_gpu))

print("CPU validation:",
      np.allclose(x_cpu, x_true, atol=1e-4))

print("GPU validation:",
      np.allclose(x_gpu, x_true, atol=1e-4))

CPU vs GPU comparison
----------------------
CPU solution error: 5787918.3938441975
GPU solution error: 1.1320862088232394e-12
CPU-GPU difference: 5787918.3938441975
CPU validation: False
GPU validation: True
